In [1]:
from dotenv import load_dotenv
load_dotenv()

from db_connection import db
from ai import get_llm, get_embedder

llm = get_llm()
embedder = get_embedder()

c:\Users\Pachara Auikim\Desktop\Graph_RAG\graph_rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


...Loading LLM model
Loaded LLM model.
...Loading Embedding model
Loaded Embedding model (BAAI/bge-m3).


## filter info

MERGE คือคำสั่ง "ถ้าไม่มีให้สร้าง ถ้ามีให้ใช้ของเดิม"

แต่ Constraint คือ "เกราะป้องกันข้อมูลซ้ำ + ตัวเร่งความเร็วในการค้นหา"
จึงจำเป็นต้องใช้คู่กันเสมอในระบบที่ใช้งานจริงครับ


create_constraints_query = """
CREATE CONSTRAINT company_name_en_unique IF NOT EXISTS
FOR (c:Company) REQUIRE c.companyEn IS UNIQUE;

CREATE CONSTRAINT coordinator_name_en_unique IF NOT EXISTS
FOR (coord:Coordinator) REQUIRE coord.nameEn IS UNIQUE;
"""

with db.get_session() as session:
    # ใน Neo4j Driver บางเวอร์ชันแนะนำให้แยกส่งทีละ statement
    session.run("CREATE CONSTRAINT company_name_en_unique IF NOT EXISTS FOR (c:Company) REQUIRE c.companyEn IS UNIQUE")
    session.run("CREATE CONSTRAINT coordinator_name_en_unique IF NOT EXISTS FOR (coord:Coordinator) REQUIRE coord.nameEn IS UNIQUE")

1. ป้องกัน Node ซ้ำจากปัญหา "แข่งกันบันทึก" (Race Condition)
Request A เช็กฐานข้อมูล -> ไม่พบ Node -> เตรียมสร้างRequest B เช็กฐานข้อมูลพร้อมกัน -> ไม่พบ Node เหมือนกัน -> เตรียมสร้างผลลัพธ์: ทั้งสองสั่งสร้าง Node พร้อมกัน ทำให้เกิด Node "Pachara Auikim" ซ้ำกัน 2 Node ในฐานข้อมูลถ้ามี Constraint: Neo4j จะล็อกระดับฐานข้อมูล ป้องกันไม่ให้สร้าง Node ซ้ำเด็ดขาด แม้จะส่งเข้ามาพร้อมกันในเสี้ยววินาทีก็ตาม (ตัวหนึ่งจะสำเร็จ อีกตัวจะดึง Node ที่สร้างเสร็จแล้วมาใช้)

2. เรื่องความเร็วและประสิทธิภาพ (Performance & Indexing)คำสั่ง MERGE จะต้องทำคำสั่ง MATCH (ค้นหา) ก่อนทุกครั้งว่ามี Node นั้นหรือยัง

- ถ้าไม่มี Constraint: Neo4j จะต้อง Scan ดูทีละ Node ทั้งฐานข้อมูล (Full Node Scan) เพื่อเช็กว่ามี nameEn: 'Pachara Auikim' แล้วหรือยัง ยิ่งข้อมูลเยอะ คำสั่ง MERGE จะยิ่งช้าลงเรื่อยๆ จนระบบอืด

- ถ้ามี Unique Constraint: Neo4j จะ สร้าง Index ให้โดยอัตโนมัติ ทำให้ค้นหาข้อมูลได้เร็วทันทีแบบ $O(1)$ ต่อให้มีเป็นล้าน Node คำสั่ง MERGE ก็ยังทำงานได้เร็วในเสี้ยววินาที

In [ ]:
add_contact_query = """
MERGE (c:Company {companyEn: $companyEn})
ON CREATE SET c.companyTh = $companyTh

MERGE (coord:Coordinator {nameEn: $nameEn})
ON CREATE SET 
  coord.nameTh = $nameTh,
  coord.nickname = $nickname,
  coord.jobTitle = $jobTitle,
  coord.phone = $phone,
  coord.email = $email

MERGE (c)-[:HAS_COORDINATOR]->(coord)
"""
with db.get_session() as session:
    session.run(add_contact_query, **result)    

## Watch later

In [ ]:
from variables import alias_company

with db.get_session() as session:
    session.run(alias_company)

In [ ]:
query= """
match (n:Company)
where n.searchText is not null
return n as company
"""
query_2= """
match (n:Company)
where n.searchText = $searchText
set 
 n.embedding = $embedding,
 n.searchText = $searchText_new
"""

target_keys = ['companyEn', 'companyTh', 'alias1', 'alias2', 'alias3']

with db.get_session() as session:
    a = session.run(query)
    for row in a.data():
        company = row['company']
        temp = []
        for key in target_keys:
            val = company.get(key, None)
            if val: temp.append(str(val).strip().lower())
            
        search_text = ", ".join(temp)
        print(search_text)
        
        searchText_new = search_text
        embed = embedder.embed_query(searchText_new)
        session.run(query_2, searchText=company['searchText'], embedding=embed, searchText_new=searchText_new)


In [ ]:
def show_index():
 with db.get_session() as session:
    a = session.run("show indexes")
    for i in a.data(): 
      print(f"type:{i['type']} name:{i['name']}")

show_index()

In [ ]:
print(len(embedder.embed_query('1')))

In [ ]:
from neo4j_graphrag.indexes import create_vector_index

INDEX_NAME = "vectorCompanyIndex"

create_vector_index(
    db.driver,
    name=INDEX_NAME,
    label="Company",               
    embedding_property="embedding", 
    dimensions=1024,            
    similarity_fn="cosine",  
    neo4j_database=os.getenv("NEO4J_DATABASE")
)


In [ ]:
from neo4j_graphrag.retrievers import VectorRetriever
from search import get_company_name

result = get_company_name("ptt")
for i in result:
  print(i.data())